# WDC Schema.org Class-Specific Subset — Domain-Targeted Extraction

This notebook:
1. Fetches the **lookup CSV** for every WDC 2024-12 class-specific subset
2. Identifies which **chunk files** contain data from our target domains
3. Downloads **only those chunks**, filters for matching quads, and deletes the download
4. Saves the filtered quads per class to `output/<ClassName>.nq`

This avoids downloading the full ~1.4 TB corpus — we only fetch the chunks we actually need.

## 1. Configuration

In [1]:
import os
import io
import gzip
import json
import time
import random
from pathlib import Path
from collections import defaultdict

import pandas as pd
import requests

# ── Target domains ──────────────────────────────────────────────
TARGET_DOMAINS = [
    # Knowledge Base
    "en.wikipedia.org",
    # Dedicated fact-checkers
    "politifact.com",
    "snopes.com",
    "factcheck.org",
    "fullfact.org",
    "africacheck.org",
    # News services
    "reuters.com",
    "apnews.com",
    # International broadcasters
    "bbc.com",
    "bbc.co.uk",
    "aljazeera.com",
    "france24.com",
    "dw.com",
    # Major newspapers
    "theguardian.com",
    "nytimes.com",
    "thetimes.com",
    "lemonde.fr",
    "cnn.com",
]

# The lookup CSV uses PLDs (pay-level domains).
# For most domains the PLD is the domain itself (bbc.com, cnn.com).
# For en.wikipedia.org, the PLD is wikipedia.org.
# We keep both the original and the PLD form.
TARGET_PLDS = set()
for d in TARGET_DOMAINS:
    TARGET_PLDS.add(d.lower())
    # Strip subdomain to get PLD: en.wikipedia.org → wikipedia.org
    # But NOT for ccTLD patterns like bbc.co.uk (already a PLD)
    parts = d.split(".")
    if len(parts) > 2:
        # Check if this is a ccTLD like co.uk, com.au, etc.
        if parts[-2] in ("co", "com", "org", "net", "edu", "gov", "ac"):
            # bbc.co.uk is already the PLD, don't strip further
            pass
        else:
            # en.wikipedia.org → wikipedia.org
            TARGET_PLDS.add(".".join(parts[-2:]).lower())

# Build patterns for line-level filtering of N-Quads
DOMAIN_PATTERNS = []
for d in TARGET_DOMAINS:
    DOMAIN_PATTERNS.append(f"://{d}/")
    DOMAIN_PATTERNS.append(f"://{d}>")
    DOMAIN_PATTERNS.append(f".{d}/")
    DOMAIN_PATTERNS.append(f".{d}>")

print(f"Target domains: {len(TARGET_DOMAINS)}")
print(f"Target PLDs (for lookup matching): {sorted(TARGET_PLDS)}")

Target domains: 18
Target PLDs (for lookup matching): ['africacheck.org', 'aljazeera.com', 'apnews.com', 'bbc.co.uk', 'bbc.com', 'cnn.com', 'dw.com', 'en.wikipedia.org', 'factcheck.org', 'france24.com', 'fullfact.org', 'lemonde.fr', 'nytimes.com', 'politifact.com', 'reuters.com', 'snopes.com', 'theguardian.com', 'thetimes.com', 'wikipedia.org']


In [2]:
# ── All 50 WDC class-specific subsets ───────────────────────────
BASE_URL = "https://data.dws.informatik.uni-mannheim.de/structureddata/2024-12/quads/classspecific"

SCHEMA_CLASSES = [
    "AdministrativeArea", "Airport", "Answer", "Book", "City",
    "ClaimReview", "CollegeOrUniversity", "Continent", "Country",
    "CreativeWork", "Dataset", "EducationalOrganization", "Event",
    "FAQPage", "GeoCoordinates", "GovernmentOrganization", "Hospital",
    "Hotel", "JobPosting", "LakeBodyOfWater", "LandmarksOrHistoricalBuildings",
    "Language", "Library", "LocalBusiness", "Mountain", "Movie",
    "Museum", "MusicAlbum", "MusicRecording", "Organization", "Painting",
    "Park", "Person", "Place", "Product", "QAPage", "Question",
    "RadioStation", "Recipe", "Restaurant", "RiverBodyOfWater", "School",
    "SearchAction", "ShoppingCenter", "SkiResort", "SportsEvent",
    "SportsTeam", "StadiumOrArena", "TVEpisode", "TelevisionStation",
]

# ── Directories ─────────────────────────────────────────────────
OUTPUT_DIR = Path("output")
TEMP_DIR = Path("wdc_temp")
LOOKUP_DIR = Path("lookups")
CHECKPOINT_FILE = Path("checkpoint_lookup.json")

OUTPUT_DIR.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)
LOOKUP_DIR.mkdir(exist_ok=True)

# ── Session ─────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({
    "User-Agent": "WDC-Research-Filter/2.0 (academic research; "
                  "contact: your-email@example.com)"
})

# ── Rate-limit settings ─────────────────────────────────────────
PAUSE_BETWEEN_DOWNLOADS = 10   # seconds between chunk downloads
MAX_RETRIES = 6

print(f"Classes: {len(SCHEMA_CLASSES)}")
print(f"Output: {OUTPUT_DIR}/")

Classes: 50
Output: output/


## 2. Helper functions

In [3]:
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f)
    return {"completed_classes": [], "completed_chunks": {}, "stats": {}}


def save_checkpoint(state):
    tmp = str(CHECKPOINT_FILE) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)


def download_with_retry(url, dest_path=None, max_retries=MAX_RETRIES):
    """
    Download a URL with retry + 429 backoff.
    If dest_path is given, saves to disk. Otherwise returns response content.
    """
    for attempt in range(max_retries):
        try:
            resp = session.get(url, timeout=300, stream=(dest_path is not None))

            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                if retry_after:
                    wait = int(retry_after) + random.uniform(5, 15)
                else:
                    wait = (120 * (2 ** attempt)) + random.uniform(0, 30)
                print(f"    ⏳ 429, waiting {wait:.0f}s (attempt {attempt+1})...")
                time.sleep(wait)
                continue

            resp.raise_for_status()

            if dest_path:
                with open(dest_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                return True
            else:
                return resp.content

        except (requests.RequestException, IOError) as e:
            wait = (60 * (2 ** attempt)) + random.uniform(0, 15)
            print(f"    ⚠ Error (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                print(f"      Retrying in {wait:.0f}s...")
                time.sleep(wait)

    return None



def line_matches_domain(line):
    line = line.rstrip()
    # Graph URI is the last <...> before the trailing " ."
    last_close = line.rfind('>')
    last_open = line.rfind('<', 0, last_close)
    if last_open == -1:
        return False
    graph_uri = line[last_open:last_close + 1]
    for pattern in DOMAIN_PATTERNS:
        if pattern in graph_uri:
            return True
    return False


print("Helpers loaded.")

Helpers loaded.


## 3. Fetch all lookup tables & identify relevant chunks

In [4]:
def fetch_lookup(cls_name):
    """
    Download the lookup CSV for a class and return a DataFrame.
    The CSV maps PLDs to chunk file numbers.
    Expected columns: pld (or domain), file (or chunk/part number).
    """
    local_path = LOOKUP_DIR / f"{cls_name}_lookup.csv"

    # Use cached version if available
    if local_path.exists() and local_path.stat().st_size > 0:
        try:
            return pd.read_csv(local_path)
        except Exception:
            pass

    url = f"{BASE_URL}/{cls_name}/{cls_name}_lookup.csv"
    content = download_with_retry(url)

    if content is None:
        print(f"  ✗ Could not download lookup for {cls_name}")
        return None

    # Save locally for caching
    with open(local_path, "wb") as f:
        f.write(content)

    try:
        return pd.read_csv(io.BytesIO(content))
    except Exception as e:
        print(f"  ✗ Could not parse lookup for {cls_name}: {e}")
        return None


# Fetch all lookups and find which chunks contain our domains
class_chunk_map = {}  # {class_name: {chunk_file: [matching_plds]}}
classes_with_no_match = []

print("Fetching lookup tables for all classes...\n")

for i, cls in enumerate(SCHEMA_CLASSES):
    print(f"[{i+1:>2}/{len(SCHEMA_CLASSES)}] {cls}...", end=" ")

    df = fetch_lookup(cls)

    if df is None:
        print("SKIP (download failed)")
        continue

    # Detect column names (varies slightly across releases)
    cols_lower = {c.lower().strip(): c for c in df.columns}

    # Find the PLD/domain column
    pld_col = None
    for candidate in ["pld", "domain", "host", "pay-level-domain", "pay_level_domain"]:
        if candidate in cols_lower:
            pld_col = cols_lower[candidate]
            break
    if pld_col is None:
        # Fall back to first column
        pld_col = df.columns[0]

    # Find the file/chunk column
    file_col = None
    for candidate in ["file", "file_lookup", "chunk", "part", "filename", "file_name"]:
        if candidate in cols_lower:
            file_col = cols_lower[candidate]
            break
    if file_col is None:
        # Fall back to second column
        file_col = df.columns[1]

    # Match our target PLDs (exact match only)
    matching = defaultdict(list)
    for _, row in df.iterrows():
        pld = str(row[pld_col]).strip().lower()
        chunk = row[file_col]

        if pld in TARGET_PLDS:
            # Normalize chunk reference to filename
            chunk_str = str(chunk).strip()
            if not chunk_str.startswith("part_"):
                chunk_str = f"part_{chunk_str}"
            if not chunk_str.endswith(".gz"):
                chunk_str = f"{chunk_str}.gz"
            matching[chunk_str].append(pld)

    if matching:
        class_chunk_map[cls] = dict(matching)
        total_plds = sum(len(v) for v in matching.values())
        print(f"✓ {total_plds} PLDs across {len(matching)} chunks")
    else:
        classes_with_no_match.append(cls)
        print(f"— no target domains found")

    # Small pause between lookup downloads
    if i < len(SCHEMA_CLASSES) - 1:
        time.sleep(1)

print(f"\n{'═' * 60}")
print(f"Classes with matches: {len(class_chunk_map)}")
print(f"Classes without matches: {len(classes_with_no_match)}")
total_chunks = sum(len(v) for v in class_chunk_map.values())
print(f"Total chunk files to download: {total_chunks}")

Fetching lookup tables for all classes...

[ 1/50] AdministrativeArea... — no target domains found
[ 2/50] Airport... — no target domains found
[ 3/50] Answer... ✓ 2 PLDs across 2 chunks
[ 4/50] Book... — no target domains found
[ 5/50] City... — no target domains found
[ 6/50] ClaimReview... ✓ 3 PLDs across 1 chunks
[ 7/50] CollegeOrUniversity... — no target domains found
[ 8/50] Continent... — no target domains found
[ 9/50] Country... — no target domains found
[10/50] CreativeWork... ✓ 3 PLDs across 3 chunks
[11/50] Dataset... — no target domains found
[12/50] EducationalOrganization... — no target domains found
[13/50] Event... ✓ 1 PLDs across 1 chunks
[14/50] FAQPage... ✓ 2 PLDs across 2 chunks
[15/50] GeoCoordinates... — no target domains found
[16/50] GovernmentOrganization... — no target domains found
[17/50] Hospital... — no target domains found
[18/50] Hotel... — no target domains found
[19/50] JobPosting... ✓ 2 PLDs across 1 chunks
[20/50] LakeBodyOfWater... — no target doma

In [5]:
# ── Summary of what we'll download ──────────────────────────────
print(f"{'Class':<35} {'Chunks':>7}  {'Matching PLDs':>15}")
print("─" * 60)
for cls, chunks in sorted(class_chunk_map.items()):
    plds = set()
    for pld_list in chunks.values():
        plds.update(pld_list)
    print(f"{cls:<35} {len(chunks):>7}  {', '.join(sorted(plds))}")

Class                                Chunks    Matching PLDs
────────────────────────────────────────────────────────────
Answer                                    2  cnn.com, wikipedia.org
ClaimReview                               1  factcheck.org, fullfact.org, politifact.com
CreativeWork                              3  factcheck.org, fullfact.org, politifact.com
Event                                     1  aljazeera.com
FAQPage                                   2  cnn.com, wikipedia.org
JobPosting                                1  theguardian.com, thetimes.com
MusicAlbum                                1  wikipedia.org
MusicRecording                            1  wikipedia.org
Organization                              6  aljazeera.com, dw.com, factcheck.org, politifact.com, thetimes.com, wikipedia.org
Person                                    8  aljazeera.com, cnn.com, dw.com, factcheck.org, france24.com, fullfact.org, politifact.com, wikipedia.org
Place                              

## 4. Download, filter, and save

In [6]:
def process_chunk(cls_name, chunk_file, output_path):
    """
    Download a chunk .gz, decompress, filter for target domain lines,
    append matches to output, delete download.
    Returns (matches, total_lines).
    """
    url = f"{BASE_URL}/{cls_name}/{chunk_file}"
    local_path = TEMP_DIR / chunk_file

    # Download
    success = download_with_retry(url, dest_path=local_path)
    if not success:
        return -1, 0

    # Filter
    matches = 0
    total_lines = 0
    matched_lines = []

    try:
        with gzip.open(local_path, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                total_lines += 1
                if line_matches_domain(line):
                    matched_lines.append(line)
                    matches += 1
    except Exception as e:
        print(f"    ⚠ Error reading {chunk_file}: {e}")
        return -1, total_lines
    finally:
        # Always delete
        if local_path.exists():
            local_path.unlink()

    # Append matches
    if matched_lines:
        with open(output_path, "a", encoding="utf-8") as out:
            out.writelines(matched_lines)

    return matches, total_lines


print("Processing function ready.")

Processing function ready.


In [7]:
# ── Save chunk plan to disk ────────────────────────────────────
import json

CHUNK_PLAN_FILE = Path("chunk_plan.json")

# Save
with open(CHUNK_PLAN_FILE, "w") as f:
    json.dump(class_chunk_map, f, indent=2)

print(f"Chunk plan saved to {CHUNK_PLAN_FILE}")
print(f"  Classes: {len(class_chunk_map)}")
total_chunks = sum(len(v) for v in class_chunk_map.values())
print(f"  Total chunks to download: {total_chunks}")

Chunk plan saved to chunk_plan.json
  Classes: 16
  Total chunks to download: 39


In [ ]:
# # ── Load chunk plan if available ───────────────────────────────
# CHUNK_PLAN_FILE = Path("chunk_plan.json")

# if CHUNK_PLAN_FILE.exists() and not class_chunk_map:
#     with open(CHUNK_PLAN_FILE, "r") as f:
#         class_chunk_map = json.load(f)
#     print(f"Loaded chunk plan from {CHUNK_PLAN_FILE}")
#     print(f"  Classes: {len(class_chunk_map)}")
#     total_chunks = sum(len(v) for v in class_chunk_map.values())
#     print(f"  Total chunks: {total_chunks}")

In [8]:
import pandas as pd
df = pd.read_csv("lookups/Answer_lookup.csv")
print(df.columns.tolist())
print(df.head(10))

['pld', 'tld', 'file_lookup']
                      pld  tld file_lookup
0           yk-groupe.com  com   part_0.gz
1               ykanda.jp   jp   part_0.gz
2               ykarc.com  com   part_0.gz
3          ykarthouse.com  com   part_0.gz
4  ykbsconsultoria.com.br   br   part_0.gz
5            ykflagman.ru   ru   part_0.gz
6          ykhukuk.com.tr   tr   part_0.gz
7         ykk-trading.com  com   part_0.gz
8               ykmbl.com  com   part_0.gz
9            yksforum.com  com   part_0.gz


In [10]:
# ── Main processing loop ───────────────────────────────────────
state = load_checkpoint()
completed_classes = set(state.get("completed_classes", []))
completed_chunks = state.get("completed_chunks", {})  # {"ClassName/chunk.gz": True}
stats = state.get("stats", {})

classes_to_process = [
    cls for cls in class_chunk_map if cls not in completed_classes
]

print(f"Classes to process: {len(classes_to_process)} "
      f"(already done: {len(completed_classes)})")
print(f"{'═' * 65}\n")

for cls_idx, cls_name in enumerate(classes_to_process):
    chunks = class_chunk_map[cls_name]
    output_path = OUTPUT_DIR / f"{cls_name}.nq"

    print(f"\n── {cls_name} ({cls_idx+1}/{len(classes_to_process)}) "
          f"— {len(chunks)} chunks ──")

    cls_matches = 0
    cls_lines = 0
    cls_failed = []

    for chunk_idx, (chunk_file, plds) in enumerate(chunks.items()):
        chunk_key = f"{cls_name}/{chunk_file}"

        # Skip already processed chunks
        if chunk_key in completed_chunks:
            print(f"  [{chunk_idx+1}/{len(chunks)}] {chunk_file} — already done, skipping")
            continue

        t0 = time.time()
        matches, total_lines = process_chunk(cls_name, chunk_file, output_path)
        elapsed = time.time() - t0

        if matches >= 0:
            cls_matches += matches
            cls_lines += total_lines
            completed_chunks[chunk_key] = True

            status = f"✓ {matches:,} matches" if matches else "  0 matches"
            print(f"  [{chunk_idx+1}/{len(chunks)}] {chunk_file}  {status}  "
                  f"({total_lines:,} lines, {elapsed:.0f}s)  "
                  f"PLDs: {', '.join(plds)}")
        else:
            cls_failed.append(chunk_file)
            print(f"  [{chunk_idx+1}/{len(chunks)}] {chunk_file}  ✗ FAILED ({elapsed:.0f}s)")

        # Save checkpoint after each chunk
        state["completed_chunks"] = completed_chunks
        save_checkpoint(state)

        # Pause between downloads
        if chunk_idx < len(chunks) - 1:
            time.sleep(PAUSE_BETWEEN_DOWNLOADS + random.uniform(0, 5))

    # Mark class as done
    if not cls_failed:
        state["completed_classes"].append(cls_name)

    # Record stats
    out_size = output_path.stat().st_size / 1024 if output_path.exists() else 0
    stats[cls_name] = {
        "matches": cls_matches,
        "lines_scanned": cls_lines,
        "chunks_processed": len(chunks) - len(cls_failed),
        "chunks_failed": len(cls_failed),
        "output_kb": round(out_size, 1),
    }
    state["stats"] = stats
    save_checkpoint(state)

    print(f"  Summary: {cls_matches:,} matches from {cls_lines:,} lines "
          f"→ {output_path} ({out_size:.1f} KB)")
    if cls_failed:
        print(f"  ⚠ Failed chunks: {cls_failed}")

    # Pause between classes
    if cls_idx < len(classes_to_process) - 1:
        pause = PAUSE_BETWEEN_DOWNLOADS * 2 + random.uniform(0, 10)
        print(f"  ⏸  Pausing {pause:.0f}s before next class...")
        time.sleep(pause)

print(f"\n{'═' * 65}")
print("All classes processed!")

Classes to process: 8 (already done: 8)
═════════════════════════════════════════════════════════════════


── Organization (1/8) — 6 chunks ──
  [1/6] part_93.gz — already done, skipping
  [2/6] part_108.gz  ✓ 1,228,468 matches  (63,197,131 lines, 1247s)  PLDs: wikipedia.org
  [3/6] part_451.gz  ✓ 826,613 matches  (12,543,886 lines, 262s)  PLDs: dw.com
  [4/6] part_680.gz  ✓ 20,875 matches  (12,532,162 lines, 275s)  PLDs: politifact.com
  [5/6] part_810.gz  ✓ 19,994 matches  (12,542,825 lines, 293s)  PLDs: factcheck.org
  [6/6] part_1515.gz  ✓ 1,819 matches  (12,530,986 lines, 238s)  PLDs: thetimes.com
  Summary: 2,097,769 matches from 113,346,990 lines → output/Organization.nq (414935.7 KB)
  ⏸  Pausing 25s before next class...

── Person (2/8) — 8 chunks ──
  [1/8] part_621.gz  ✓ 27,285 matches  (12,559,574 lines, 290s)  PLDs: fullfact.org
  [2/8] part_784.gz    0 matches  (12,546,247 lines, 295s)  PLDs: wikipedia.org
  [3/8] part_831.gz  ✓ 20,875 matches  (12,531,372 lines, 343s)  

## 5. Summary

In [11]:
# ── Final summary ──────────────────────────────────────────────
state = load_checkpoint()
stats = state.get("stats", {})

if stats:
    summary_rows = []
    for cls_name, s in sorted(stats.items()):
        if s["matches"] > 0:
            summary_rows.append({
                "Class": cls_name,
                "Matches": s["matches"],
                "Lines Scanned": s["lines_scanned"],
                "Chunks": s["chunks_processed"],
                "Failed": s["chunks_failed"],
                "Output (KB)": s["output_kb"],
            })

    df_summary = pd.DataFrame(summary_rows)
    if not df_summary.empty:
        print("\nClasses with matching quads:\n")
        print(df_summary.to_string(index=False))
        print(f"\nTotal matches: {df_summary['Matches'].sum():,}")
        print(f"Total output:  {df_summary['Output (KB)'].sum():.1f} KB")
    else:
        print("No matches found in any class.")

    # Classes with 0 matches
    zero_classes = [c for c, s in stats.items() if s["matches"] == 0]
    if zero_classes:
        print(f"\nClasses with 0 matches: {', '.join(sorted(zero_classes))}")
else:
    print("No stats recorded yet — run the processing cell first.")


Classes with matching quads:

       Class  Matches  Lines Scanned  Chunks  Failed  Output (KB)
      Answer      551       25068272       2       0        140.3
 ClaimReview    75924        3919703       1       0      15965.4
CreativeWork    74967       37631469       3       0      15762.6
       Event      623       12529590       1       0        179.8
     FAQPage      551       25096515       2       0        140.3
  JobPosting      718       12536476       1       0        211.3
Organization  2097769      113346990       6       0     414935.7
      Person  2389601      100454336       8       0     625080.5
       Place    12770       41581019       3       0       2183.9
     Product     5423       12533384       1       0       1246.8
    Question      551       25659063       2       0        140.3
SearchAction   299195       62849976       5       0      75949.5
 SportsEvent    11719       12535401       1       0       1844.1
  SportsTeam    11497       16211819       1 

In [12]:
# ── List output files ──────────────────────────────────────────
print("\nOutput files:\n")
for f in sorted(OUTPUT_DIR.glob("*.nq")):
    size = f.stat().st_size
    if size > 0:
        # Count lines
        with open(f, "r") as fh:
            n_lines = sum(1 for _ in fh)
        if size > 1024 * 1024:
            print(f"  {f.name:<40} {size / (1024*1024):.1f} MB  ({n_lines:,} quads)")
        else:
            print(f"  {f.name:<40} {size / 1024:.1f} KB  ({n_lines:,} quads)")


Output files:

  Answer.nq                                140.3 KB  (551 quads)
  ClaimReview.nq                           15.6 MB  (75,924 quads)
  CreativeWork.nq                          15.4 MB  (74,967 quads)
  Event.nq                                 179.8 KB  (623 quads)
  FAQPage.nq                               140.3 KB  (551 quads)
  JobPosting.nq                            211.3 KB  (718 quads)
  Organization.nq                          405.2 MB  (2,109,799 quads)
  Person.nq                                610.4 MB  (2,389,601 quads)
  Place.nq                                 2.1 MB  (12,770 quads)
  Product.nq                               1.2 MB  (5,423 quads)
  Question.nq                              140.3 KB  (551 quads)
  SearchAction.nq                          74.2 MB  (299,195 quads)
  SportsEvent.nq                           1.8 MB  (11,719 quads)
  SportsTeam.nq                            1.8 MB  (11,497 quads)


## 6. Quick peek at results

In [15]:
# Show first 5 lines from each non-empty output file
for f in sorted(OUTPUT_DIR.glob("*.nq")):
    if f.stat().st_size > 0:
        print(f"\n{'─' * 60}")
        print(f"  {f.name}")
        print(f"{'─' * 60}")
        with open(f, "r") as fh:
            for i, line in enumerate(fh):
                if i >= 5:
                    print(f"  ... (truncated)")
                    break
                # Truncate long lines for display
                display = line.strip()
                if len(display) > 120:
                    display = display[:120] + "..."
                print(f"  {display}")


────────────────────────────────────────────────────────────
  Answer.nq
────────────────────────────────────────────────────────────
  <https://www.mg4law.com/#Organization> <http://schema.org/areaServed> "https://en.wikipedia.org/wiki/Conyers,_Georgia" <...
  <https://www.mg4law.com/#Organization> <http://schema.org/areaServed> "https://en.wikipedia.org/wiki/Covington,_Georgia"...
  <https://www.mg4law.com/#Organization> <http://schema.org/areaServed> "https://en.wikipedia.org/wiki/Atlanta" <https://w...
  <https://www.mgmotor.co.in/vehicles/comet-ev-electric-car-in-india> <http://schema.org/sameAs> <https://en.wikipedia.org...
  <https://www.mgmotor.co.in/vehicles/mghector-plus-6-seater> <http://schema.org/sameAs> <https://en.wikipedia.org/wiki/MG...
  ... (truncated)

────────────────────────────────────────────────────────────
  ClaimReview.nq
────────────────────────────────────────────────────────────
  _:n8988ab806129453cbf92733aa7a2e832xb0 <http://schema.org/claimReviewed> "&

In [16]:
# ── Merge + deduplicate all .nq files, collect URI stats ────────
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter

OUTPUT_DIR = Path("output")
MERGED_FILE = OUTPUT_DIR / "merged.nq"

def _extract_graph_uri(line):
    line = line.rstrip()
    last_close = line.rfind('>')
    if last_close == -1:
        return None
    last_open = line.rfind('<', 0, last_close)
    if last_open == -1:
        return None
    return line[last_open + 1:last_close]

def _parse_host(uri):
    try:
        host = urlparse(uri).netloc.lower()
        parts = host.split(".")
        if len(parts) >= 3 and parts[-2] in ("co", "com", "org", "net", "edu", "gov", "ac"):
            reg = ".".join(parts[-3:])
        else:
            reg = ".".join(parts[-2:]) if len(parts) >= 2 else host
        return host, reg
    except Exception:
        return None, None

seen_hashes = set()
uri_records = []          # (class_name, full_uri, host, registered_domain)
per_class_counts = {}
total_dups = 0

print(f"{'Class':<20} {'total':>10}  {'unique':>10}  {'dups':>8}")
print("─" * 55)

with open(MERGED_FILE, "w", encoding="utf-8") as out:
    for nq_file in sorted(OUTPUT_DIR.glob("*.nq")):
        if nq_file.name == "merged.nq":
            continue
        cls_name = nq_file.stem
        cls_total = cls_unique = 0

        with open(nq_file, encoding="utf-8") as f:
            for line in f:
                cls_total += 1
                h = hash(line)
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    out.write(line)
                    cls_unique += 1
                    uri = _extract_graph_uri(line)
                    if uri:
                        host, reg = _parse_host(uri)
                        if host:
                            uri_records.append((cls_name, uri, host, reg))
                else:
                    total_dups += 1

        per_class_counts[cls_name] = {"total": cls_total, "unique": cls_unique}
        print(f"{cls_name:<20} {cls_total:>10,}  {cls_unique:>10,}  {total_dups:>8,}")

total_read = sum(v["total"]  for v in per_class_counts.values())
total_uniq = sum(v["unique"] for v in per_class_counts.values())
print(f"\n{'='*55}")
print(f"Total read:    {total_read:>10,}")
print(f"Duplicates:    {total_dups:>10,}  ({total_dups/total_read*100:.2f}%)")
print(f"Unique quads:  {total_uniq:>10,}")
print(f"Merged → {MERGED_FILE}  ({MERGED_FILE.stat().st_size / 1e6:.1f} MB)")


Class                     total      unique      dups
───────────────────────────────────────────────────────
Answer                      551         551         0
ClaimReview              75,924      75,898        26
CreativeWork             74,967           0    74,993
Event                       623         598    75,018
FAQPage                     551           0    75,569
JobPosting                  718         718    75,569
Organization          2,109,799   2,067,365   118,003
Person                2,389,601   1,892,252   615,352
Place                    12,770      11,497   616,625
Product                   5,423           0   622,048
Question                    551           0   622,599
SearchAction            299,195     104,129   817,665
SportsEvent              11,719         222   829,162
SportsTeam               11,497           0   840,659

Total read:     4,993,889
Duplicates:       840,659  (16.83%)
Unique quads:   4,153,230
Merged → output/merged.nq  (982.2 MB)


In [17]:
# ── Build stats DataFrame ────────────────────────────────────────
import pandas as pd

df_uris = pd.DataFrame(uri_records, columns=["class", "uri", "host", "registered_domain"])

print(f"Unique quads with parseable source URI: {len(df_uris):,}")
print(f"Unique hosts (subdomains):              {df_uris['host'].nunique():,}")
print(f"Unique registered domains:              {df_uris['registered_domain'].nunique():,}")
print()

# Quads per registered domain
domain_totals = df_uris["registered_domain"].value_counts()
print("Quads per registered domain:")
print(f"  {'domain':<35} {'quads':>9}  {'%':>6}")
print(f"  {'-'*52}")
for domain, count in domain_totals.items():
    pct = count / len(df_uris) * 100
    print(f"  {domain:<35} {count:>9,}  {pct:>5.1f}%")


Unique quads with parseable source URI: 4,153,230
Unique hosts (subdomains):              28
Unique registered domains:              10

Quads per registered domain:
  domain                                  quads       %
  ----------------------------------------------------
  cnn.com                             1,915,302   46.1%
  wikipedia.org                       1,240,187   29.9%
  dw.com                                826,324   19.9%
  factcheck.org                          90,145    2.2%
  fullfact.org                           38,215    0.9%
  politifact.com                         20,875    0.5%
  aljazeera.com                          12,410    0.3%
  france24.com                            7,842    0.2%
  thetimes.com                            1,862    0.0%
  theguardian.com                            68    0.0%


In [19]:
# ── Subdomain distribution ───────────────────────────────────────
from collections import Counter

print(f"{'='*65}")
print(f"Subdomains per registered domain  (merged, deduplicated quads)")
print(f"{'='*65}\n")

domain_hosts = df_uris.groupby("registered_domain")["host"].apply(list)

for domain, hosts in sorted(domain_hosts.items(), key=lambda x: -len(x)):
    host_counts = Counter(hosts)
    print(f"{domain}  ({len(hosts):,} quads total)")
    print(f"  {'subdomain':<45} {'count':>8}  {'%':>6}")
    print(f"  {'-'*62}")
    for host, count in host_counts.most_common():
        pct = count / len(hosts) * 100
        print(f"  {host:<45} {count:>8,}  {pct:>5.1f}%")
    print()


Subdomains per registered domain  (merged, deduplicated quads)

aljazeera.com  (12,410 quads total)
  subdomain                                        count       %
  --------------------------------------------------------------
  www.aljazeera.com                               12,243   98.7%
  interactive.aljazeera.com                          161    1.3%
  mediaview.aljazeera.com                              6    0.0%

cnn.com  (1,915,302 quads total)
  subdomain                                        count       %
  --------------------------------------------------------------
  cnnespanol.cnn.com                            1,772,239   92.5%
  amp.cnn.com                                     88,967    4.6%
  arabic.cnn.com                                  33,116    1.7%
  cnne-stage.cnn.com                              11,241    0.6%
  commercial.cnn.com                               8,670    0.5%
  cnne-test.cnn.com                                  568    0.0%
  sponsorcontent.cnn

In [ ]:
# ── Schema class breakdown per domain + top source pages ─────────
# numbers include other classes existing with the main classes (how many quads came from each main class file per domain "which includes other classes if exist in same url")

# Class × registered domain pivot
print(f"{'='*70}")
print("Schema.org class breakdown per registered domain (quad counts)")
print(f"{'='*70}\n")

pivot = (
    df_uris.groupby(["registered_domain", "class"])
    .size()
    .unstack(fill_value=0)
)
pivot.insert(0, "TOTAL", pivot.sum(axis=1))
pivot = pivot.sort_values("TOTAL", ascending=False)
print(pivot.to_string())

# Top 20 most-annotated source pages
print(f"\n{'='*70}")
print("Top 20 source pages by quad count")
print(f"{'='*70}")
print(f"  {'quads':>6}  url")
print(f"  {'-'*66}")
top_pages = df_uris.groupby("uri").size().sort_values(ascending=False).head(20)
for uri, count in top_pages.items():
    display = uri if len(uri) <= 80 else uri[:77] + "..."
    print(f"  {count:>6,}  {display}")


Schema.org class breakdown per registered domain (quad counts)

class                TOTAL  Answer  ClaimReview  Event  JobPosting  Organization   Person  Place  SearchAction  SportsEvent
registered_domain                                                                                                          
cnn.com            1915302     551            0      0           0             0  1821834      0         92917            0
wikipedia.org      1240187       0            0      0           0       1228468        0  11497             0          222
dw.com              826324       0            0      0           0        826321        3      0             0            0
factcheck.org        90145       0        19994      0           0             0    70151      0             0            0
fullfact.org         38215       0        35029      0           0             0        0      0          3186            0
politifact.com       20875       0        20875      0           0  

In [27]:
# ── All schema.org types × domain (from rdf:type triples in merged) ──
# how many type declarations of each class per domain

RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
type_records = []

import re

def normalize_schema_type(uri):
    return re.sub(r"^https?://(?:www\.)?schema\.org/", "", uri)


with open(MERGED_FILE, encoding="utf-8") as f:
    for line in f:
        if RDF_TYPE not in line:
            continue
        parts = line.split()
        if len(parts) < 4 or parts[1] != RDF_TYPE:
            continue
        uri = _extract_graph_uri(line)
        if not uri:
            continue
        _, reg = _parse_host(uri)
        if not reg:
            continue
        schema_type = normalize_schema_type(parts[2].strip("<>"))
        type_records.append((reg, schema_type))

df_types = pd.DataFrame(type_records, columns=["registered_domain", "schema_type"])

print(f"Type declarations: {len(df_types):,}")
print(f"Distinct schema.org types: {df_types['schema_type'].nunique()}\n")

pivot_all = (
    df_types.groupby(["registered_domain", "schema_type"])
    .size()
    .unstack(fill_value=0)
)
pivot_all.insert(0, "TOTAL", pivot_all.sum(axis=1))
pivot_all = pivot_all.sort_values("TOTAL", ascending=False)
print(pivot_all.to_string())

# Top 20 most-annotated source pages (unchanged)
print(f"\n{'='*70}")
print("Top 20 source pages by quad count")
print(f"{'='*70}")
print(f"  {'quads':>6}  url")
print(f"  {'-'*66}")
top_pages = df_uris.groupby("uri").size().sort_values(ascending=False).head(20)
for uri, count in top_pages.items():
    display = uri if len(uri) <= 80 else uri[:77] + "..."
    print(f"  {count:>6,}  {display}")


Type declarations: 816,664
Distinct schema.org types: 43

schema_type         TOTAL  AggregateOffer  Answer  Article  BlogPosting  BreadcrumbList  BroadcastEvent  Claim  ClaimReview  CollectionPage  CreativeWork  EntryPoint  Event  FAQPage  ImageGallery  ImageObject  JobPosting  ListItem  LiveBlogPosting  MediaObject  NewsArticle  NewsMediaOrganization  Offer  Organization  Person  Place  PostalAddress  Product  ProfilePage  PropertyValue  PropertyValueSpecification  Question  Rating  ReadAction  SearchAction  SearchResultsPage  SiteNavigationElement  SpeakableSpecification  SportsEvent  SportsTeam  VideoObject  WebPage  WebPageElement  WebSite
registered_domain                                                                                                                                                                                                                                                                                                                                          

In [21]:
# ── Unique annotated pages per domain ────────────────────────────
ann = (
    df_uris.groupby("registered_domain")
    .agg(
        quads=("uri", "count"),
        pages=("uri", "nunique"),
    )
    .sort_values("quads", ascending=False)
)
ann["quads_per_page"] = (ann["quads"] / ann["pages"]).round(1)

print(f"{'domain':<35} {'pages':>8}  {'quads':>10}  {'quads/page':>11}")
print("─" * 68)
for domain, row in ann.iterrows():
    print(f"{domain:<35} {row['pages']:>8,}  {row['quads']:>10,}  {row['quads_per_page']:>11.1f}")


domain                                 pages       quads   quads/page
────────────────────────────────────────────────────────────────────
cnn.com                             18,825.0  1,915,302.0        101.7
wikipedia.org                       66,739.0  1,240,187.0         18.6
dw.com                              41,245.0   826,324.0         20.0
factcheck.org                        1,602.0    90,145.0         56.3
fullfact.org                         1,158.0    38,215.0         33.0
politifact.com                         725.0    20,875.0         28.8
aljazeera.com                          334.0    12,410.0         37.2
france24.com                           237.0     7,842.0         33.1
thetimes.com                           118.0     1,862.0         15.8
theguardian.com                          2.0        68.0         34.0


In [30]:
# ── Actual schema.org types present in each class file ───────────
import re
from collections import Counter
from pathlib import Path

OUTPUT_DIR = Path("output")
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"

def normalize_schema_type(uri):
    uri = re.sub(r"^https?://(?:www\.)?schema\.org/", "", uri)
    uri = uri.replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:")
    return uri

print(f"{'='*65}")
print("Schema.org types found inside each class file")
print(f"{'='*65}")

for nq_file in sorted(OUTPUT_DIR.glob("*.nq")):
    if nq_file.name == "merged.nq":
        continue
    type_counts = Counter()
    with open(nq_file, encoding="utf-8") as f:
        for line in f:
            if RDF_TYPE not in line:
                continue
            parts = line.split()
            if len(parts) >= 4 and parts[1] == RDF_TYPE:
                label = normalize_schema_type(parts[2].strip("<>"))
                type_counts[label] += 1

    print(f"\n{nq_file.stem}  ({sum(type_counts.values()):,} type triples)")
    print(f"  {'type':<55} {'count':>8}")
    print(f"  {'-'*65}")
    for t, count in type_counts.most_common(20):
        print(f"  {t:<55} {count:>8,}")


Schema.org types found inside each class file

Answer  (112 type triples)
  type                                                       count
  -----------------------------------------------------------------
  ImageObject                                                   23
  Product                                                       21
  AggregateOffer                                                21
  Offer                                                         19
  Organization                                                  11
  Question                                                       4
  Answer                                                         4
  NewsMediaOrganization                                          2
  FAQPage                                                        1
  BreadcrumbList                                                 1
  ListItem                                                       1
  NewsArticle                                         

In [31]:
# ── Actual schema.org types in merged (deduplicated) quads ───────
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
type_counts = Counter()

with open(MERGED_FILE, encoding="utf-8") as f:
    for line in f:
        if RDF_TYPE not in line:
            continue
        parts = line.split()
        if len(parts) >= 4 and parts[1] == RDF_TYPE:
            label = normalize_schema_type(parts[2].strip("<>"))
            type_counts[label] += 1

print(f"Type triples in merged file: {sum(type_counts.values()):,}\n")
print(f"  {'type':<55} {'count':>8}")
print(f"  {'-'*65}")
for t, count in type_counts.most_common(30):
    print(f"  {t:<55} {count:>8,}")


Type triples in merged file: 816,664

  type                                                       count
  -----------------------------------------------------------------
  Organization                                             263,128
  ImageObject                                              176,129
  Article                                                   69,588
  Person                                                    61,283
  WebPage                                                   59,857
  NewsArticle                                               58,059
  NewsMediaOrganization                                     30,890
  VideoObject                                               16,267
  WebPageElement                                            15,085
  ListItem                                                   9,429
  MediaObject                                                9,123
  SiteNavigationElement                                      7,020
  BlogPosting          